# FlashEats Operational Bottleneck Diagnosis & Dependable KPI Pipeline
### FDE Data Foundations (Classes 4–8) | Interactive Walkthrough Notebook
**Author:** Harshit Tiwari (Roll Number: `24BCS10277`)  
**Role:** Forward Deployed Engineer (FDE)  
**GitHub Repo:** [https://github.com/X-DIABLO-X/FDE-2-Assignement](https://github.com/X-DIABLO-X/FDE-2-Assignement)

---

## 1. Multi-Mode Ingestion & Completeness Proofs (Class 5)
Extract data across 4 disparate systems:
1. SQL RDBMS (`orders_platform.db`)
2. REST API / JSON Stream (`dispatch_telemetry_stream.json`)
3. Merchant KDS CSV Logs (`kds_merchant_prep.csv`)
4. Customer Support Zendesk JSON (`customer_support_tickets.json`)

Guarantees raw data immutability and proves mathematical completeness via row counts and GMV control totals.

In [ ]:
from src.ingest.ingest_manager import IngestionManager

ingest_mgr = IngestionManager()
proof = ingest_mgr.run_all()
print("Completeness verified:", proof["completeness_verified"])
print("GMV Control Total: $", proof["control_totals"]["gmv_subtotal_cents"] / 100)

## 2. Data Profiling, Validation & Quarantine Engine (Class 6)
FDE Principle: **Zero silent dropping of data.**
- Calibrates recoverable anomalies (POS clock drift, missing bump bars, network jitter).
- Isolates severe corruptions to `quarantine_audit.parquet` under circuit-breaker monitoring.

In [ ]:
from src.validate.validator import DataValidator

validator = DataValidator()
val_summary = validator.validate_and_calibrate()
print("Quarantine Summary:", val_summary["quarantine_summary"])
print("Calibration Audit:", val_summary["calibration_audit"])

## 3. Workflow State Machine & 5-Stage Delay Decomposition (Class 7)
Reconstructs order lifecycle:
`SUBMITTED -> CONFIRMED -> PREP_STARTED -> FOOD_READY -> COURIER_ASSIGNED -> ARRIVED_STORE -> PICKED_UP -> DELIVERED`

Decomposes delay into 5 stages to isolate the primary bottleneck.

In [ ]:
from src.model.state_machine import WorkflowStateMachine

sm = WorkflowStateMachine()
fact_df = sm.build_fact_order_lifecycle()
fact_df[["order_id", "total_otd_min", "rider_store_wait_min", "stage2_prep_overrun_min", "primary_delay_bottleneck"]].head(10)

## 4. Gold Analytical Marts & Counterfactual Intervention Simulations (Class 7 & 8)
Computes North Star operational KPIs and simulates 3 interventions:
1. Baseline: Immediate Courier Dispatch
2. Intervention 1: Staged Offset Dispatch (Courier arrival JIT before food ready)
3. Intervention 2: Queue-Aware Dynamic Prep Buffers
4. Intervention 3: Combined Strategy

In [ ]:
from src.marts.gold_marts import GoldMartsManager

marts = GoldMartsManager()
kpis, clusters, sims = marts.run_all()

## 5. Executive Dashboard & Evidence Exporter
Renders the executive terminal briefing and exports the final evidence table.

In [ ]:
from src.reporting.dashboard import FlashEatsReporter

reporter = FlashEatsReporter()
reporter.print_terminal_dashboard()